# pLac Promoter — Statistical Mechanics & Stochastic Simulation

This notebook walks through two advanced models of the *E. coli* **lac promoter**:

1. **Thermodynamic model** — partition-function approach from statistical mechanics  
2. **Gillespie SSA** — exact stochastic simulation capturing single-cell noise

Both are implemented in `tibs.plac` and follow the same `ODEModel` conventions
used elsewhere in the repo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# works whether tibs is pip-installed or run from the repo root
try:
    from tibs.plac import (
        ThermodynamicPromoter, PLac, GillespiePLac,
        plot_dose_response, plot_stochastic_traces,
    )
except ImportError:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path().resolve().parent / "src"))
    from tibs.plac import (
        ThermodynamicPromoter, PLac, GillespiePLac,
        plot_dose_response, plot_stochastic_traces,
    )

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 1  Thermodynamic Model

We enumerate five promoter microstates (empty, RNAP-bound, LacI at O1,
LacI at O2, DNA-looped) and assign Boltzmann weights based on binding
free energies.  The probability of transcription is:

$$p_{\text{active}} = \frac{w_{\text{RNAP}}}{Z}$$

where $Z = 1 + w_{\text{RNAP}} + w_{O1} + w_{O2} + w_{\text{loop}}$.

IPTG reduces the number of active (DNA-competent) repressors:

$$R_A = R_{\text{total}} \left( \frac{1}{1 + [\text{IPTG}]/K_d} \right)^2$$

In [ ]:
# Dose-response for different repressor copy numbers
fig = plot_dose_response(repressor_counts=[1, 5, 10, 50, 200])
plt.show()

In [ ]:
# Inspect the microstate probabilities at a particular IPTG concentration
tp = ThermodynamicPromoter(R_total=10)

for iptg_uM, iptg_M in [(0, 0), (0.1, 1e-7), (10, 1e-5), (1000, 1e-3)]:
    probs = tp.state_probabilities(iptg_M)
    print(f"\n[IPTG] = {iptg_uM} µM")
    for state, p in probs.items():
        bar = '█' * int(p * 50)
        print(f"  {state:15s}  {p:.4f}  {bar}")

### 1.1  Effect of operator binding energy

What if we engineer a weaker or stronger operator? The thermodynamic
framework lets us predict the dose–response shift directly.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

# vary the O1 binding energy
for dE, style in [(-10, '--'), (-13, '-.'), (-15.3, '-'), (-18, ':')]:
    tp = ThermodynamicPromoter(R_total=10, dE_R_O1=dE)
    iptg, fc = tp.dose_response()
    ax.semilogx(iptg * 1e6, fc, style, lw=2, label=f'ΔεO1 = {dE} kBT')

ax.set(xlabel='[IPTG] (µM)', ylabel='fold-change',
       title='operator strength tunes the induction curve')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

---
## 2  ODE Model — Deterministic Dynamics

`PLac` wraps the thermodynamic model into a two-variable ODE
(mRNA, protein) that subclasses `ODEModel` from the repo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (iptg, label) in zip(axes, [(0, '0 µM'), (1e-5, '10 µM'), (1e-3, '1 mM')]):
    m = PLac(IPTG=iptg)
    res = m.simulate(t_span=(0, 300), dt=0.1)
    ax.plot(res.t, res.x[:, 0], label='mRNA', lw=1.5)
    ax.plot(res.t, res.x[:, 1], label='protein', lw=1.5)
    ax.set(xlabel='time (min)', title=f'[IPTG] = {label}')
    ax.legend(fontsize=9)

axes[0].set_ylabel('molecules (a.u.)')
fig.suptitle('pLac ODE dynamics at three induction levels', fontsize=13)
fig.tight_layout()
plt.show()

---
## 3  Gillespie SSA — Single-Cell Stochasticity

Gene expression is inherently noisy at the molecular level. The
Gillespie algorithm simulates the exact stochastic dynamics by
sampling exponential waiting times between discrete reaction events.

### 3.1  Single-cell traces

In [ ]:
g = GillespiePLac(IPTG=1e-3, seed=42)
traces = g.run_ensemble(n_cells=50, t_end=300.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

plot_stochastic_traces(traces, species=0, n_show=20, ax=axes[0])
axes[0].set_title('mRNA — 20 cells (1 mM IPTG)')

plot_stochastic_traces(traces, species=1, n_show=20, ax=axes[1])
axes[1].set_title('Protein — 20 cells (1 mM IPTG)')

fig.tight_layout()
plt.show()

### 3.2  Population distribution at steady state

Collect the protein count at the end of each trajectory to build
a histogram — the single-cell distribution of expression levels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (iptg, label) in zip(axes, [(1e-7, '0.1 µM'), (1e-5, '10 µM'), (1e-3, '1 mM')]):
    g = GillespiePLac(IPTG=iptg, seed=7)
    ens = g.run_ensemble(n_cells=300, t_end=300.0)
    finals = [X[-1, 1] for _, X in ens]
    
    ax.hist(finals, bins=30, color='#2563eb', alpha=0.7, edgecolor='white')
    mu, sigma = np.mean(finals), np.std(finals)
    cv = sigma / mu if mu > 0 else float('nan')
    ax.axvline(mu, color='#dc2626', ls='--', lw=2, label=f'mean={mu:.1f}')
    ax.set(xlabel='protein / cell', title=f'[IPTG]={label}  (CV={cv:.2f})')
    ax.legend(fontsize=9)

axes[0].set_ylabel('# cells')
fig.suptitle('steady-state protein distributions (n=300 cells)', fontsize=13)
fig.tight_layout()
plt.show()

### 3.3  Noise scales inversely with mean expression

A classic prediction: for a simple birth-death process, the coefficient
of variation $CV = \sigma / \mu$ scales as $1/\sqrt{\mu}$. Let's check
whether that holds across IPTG concentrations.

In [ ]:
iptg_scan = np.logspace(-7.5, -2.5, 10)
means, cvs = [], []

for iptg in iptg_scan:
    g = GillespiePLac(IPTG=iptg, seed=99)
    ens = g.run_ensemble(n_cells=200, t_end=300.0)
    finals = np.array([X[-1, 1] for _, X in ens], dtype=float)
    mu = finals.mean()
    means.append(mu)
    cvs.append(finals.std() / mu if mu > 0 else np.nan)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.loglog(iptg_scan * 1e6, means, 'o-', color='#2563eb', lw=2)
ax1.set(xlabel='[IPTG] (µM)', ylabel='mean protein / cell',
        title='mean expression vs IPTG')

ax2.loglog(means, cvs, 's', color='#dc2626', ms=8)
# overlay 1/sqrt(n) reference
m_ref = np.linspace(min(means) * 0.5, max(means) * 2, 100)
ax2.loglog(m_ref, 1.0 / np.sqrt(m_ref), '--', color='gray', label=r'$1/\sqrt{\mu}$')
ax2.set(xlabel='mean protein', ylabel='CV', title='noise vs mean')
ax2.legend()

fig.tight_layout()
plt.show()

---
## 4  Regime Classification

We can use the repo's `classify_timeseries` to label the ODE
trajectories automatically.

In [ ]:
try:
    from tibs.analysis import classify_timeseries
    for iptg in [0, 1e-6, 1e-4, 1e-3]:
        m = PLac(IPTG=iptg)
        res = m.simulate(t_span=(0, 500), dt=0.1)
        regime = classify_timeseries(res.x)
        print(f"[IPTG] = {iptg:.0e} M  →  {regime}")
except ImportError:
    print("(tibs.analysis not available — skipping regime classification)")

---
## Next Steps

Some ideas for extending this model:

- **LacY positive feedback** — add a third species (permease) that increases
  intracellular IPTG, creating bistability at intermediate inducer levels.
- **Toggle switch** — combine pLac with pTet in a mutual-repression circuit.
- **Promoter engineering** — use the thermodynamic model to predict how
  mutating operator sequences shifts the dose–response curve.
- **Fit to data** — calibrate binding energies against experimental
  dose–response data from the literature.